![image](image.jpg)

The fast food industry represents one of the most dynamic sectors in the stock market, with companies ranging from global giants like McDonald's to innovative newcomers like Luckin Coffee. In this project, you'll create an interactive line chart to visualize historical stock prices for 10 major fast-food companies. Building interactive dashboards is a critical skill in modern business, as these tools help analysts, portfolio managers, and executives make informed, data-driven decisions.

## Dataset Summary

Your primary dataset, `companies.csv`, contains historical stock market data from Yahoo Finance. This dataset captures daily trading activity from major players across the fast food sector. It includes stock data from these industry leaders:

| Ticker | Company | Description |
|--------|---------|-------------|
| **BRK-A** | Berkshire Hathaway Inc. | Financial conglomerate with major fast food investments |
| **DNUT** | Krispy Kreme, Inc. | Specialty donut and coffee retailer |
| **DPZ** | Domino's Pizza, Inc. | Global pizza delivery leader |
| **LKNCY** | Luckin Coffee Inc. | Chinese coffee chain competitor |
| **MCD** | McDonald's Corporation | World's largest fast food restaurant chain |
| **PZZA** | Papa John's International | Pizza delivery and takeout specialist |
| **QSR** | Restaurant Brands International | Parent of Burger King, Tim Hortons, and Popeyes |
| **SBUX** | Starbucks Corporation | Global coffeehouse chain |
| **WEN** | The Wendy's Company | Premium burger restaurant chain |
| **YUM** | Yum! Brands, Inc. | Parent of KFC, Taco Bell, and Pizza Hut |

**Note:** A ticker is a unique stock symbol used to identify companies on exchanges (e.g., "MCD" for McDonald's, "SBUX" for Starbucks).


## Data Structure

`companies.csv`
| Column | Description |
|--------|-------------|
| `date` | Trading date |
| `open` | Opening price for the trading session |
| `high` | Highest price during the day |
| `low` | Lowest price during the day |
| `close` | Closing price at market close |
| `adj_close` | Price adjusted for dividends and stock splits |
| `volume` | Number of shares traded |
| `company_ticker` | Company stock symbol identifier |

# Objectives
**Create an interactive stock price visualization using Plotly to help clients analyze fast food industry trends.**
- Build an interactive line chart and save it as `fig`. Include 4 time-range buttons (`1M`, `6M`, `1Y`, `3Y`) to allow users to filter data by different periods.
- Add two dropdown menus: the first should allow selection of specific companies using the `company_ticker` column, and the second should enable choosing which price metric to display (such as `open`, `close`, `high`, `low`, etc.).
- Apply styling, hover effects, and annotations of your choice to create a professional appearance. ⚠️ This styling component will not be tested.

In [75]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

In [76]:
#import data
orig_df = pd.read_csv('companies.csv')

#inspect data
display(orig_df.head(5))
display(orig_df.info())

,date,open,high,low,close,adj_close,volume,company_ticker
0,2019-01-02,302000.000000,306255.000000,301880.000000,304057.000000,304057.000000,400,BRK-A
1,2019-01-02,39.639999,40.660000,38.919998,40.130001,36.164330,370200,PZZA
2,2019-01-02,245.589996,245.589996,240.240005,243.300003,228.573074,444700,DPZ
3,2019-01-02,51.610001,52.060001,50.779999,51.419998,42.610756,1939100,QSR
4,2019-01-02,91.089996,91.540001,90.349998,91.440002,82.342484,1743400,YUM


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11857 entries, 0 to 11856
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   date            11857 non-null  object 
 1   open            11857 non-null  float64
 2   high            11857 non-null  float64
 3   low             11857 non-null  float64
 4   close           11857 non-null  float64
 5   adj_close       11857 non-null  float64
 6   volume          11857 non-null  int64  
 7   company_ticker  11857 non-null  object 
dtypes: float64(5), int64(1), object(2)
memory usage: 741.2+ KB


None

There are 11,857 data points & eight variables. There are no missing values, the data types seem appropriate; however, the 'date' variable could be converted to a datetime format.  
The data spans 2019-01-02 through 2023-12-29; nearly five years.

In [77]:
#convert 'date' to datetime format
orig_df['date'] = pd.to_datetime(orig_df['date'])

#explore data
#display(orig_df['date'].describe())

#save cleaned data
df = orig_df.copy()

Firstly, want to build an interactive line chart that includes four time-range buttons that will allow the user to view the plot across various time periods. Although multiple metrics will be explored according to a dropdown menu, the default metric will be the adjusted closing price (the 'adj_close' variable).
- They include the last one month, the last six months, the last year, & the last three years.

In [78]:
#Construct the figure, add data from each company individually
fig = go.Figure()
for comp in df['company_ticker'].unique():
    comp_df = df[df['company_ticker'] == comp]
    #trace = px.line(comp_df, x='date', y='adj_close').data[0]
    #trace.visible = (comp == default_ticker) -- OPTIONAL (default company only → cleaner first view)
    #fig.add_trace(trace)
    fig.add_trace(px.line(comp_df, x='date', y='adj_close').data[0])

In [79]:
#construct time-range buttons
time_range_buttons = [{'count':1, 'step':'month', 'stepmode':'todate', 'label':'1M'},
                   {'count':6, 'step':'month', 'stepmode':'todate', 'label':'6M'},
                   {'count':1, 'step':'year', 'stepmode':'todate', 'label':'1Y'},
                   {'count':3, 'step':'year', 'stepmode':'todate', 'label':'3Y'}]

Next, want to add two dropdown menus that enable the selection of specific companies & what price metric to display ('open', 'close', 'high', 'low', etc.).

In [80]:
#Construct dropdown menus -- Company tickers

#create 10 lists for 10 companies for the nested 'visible' parameter
TF_vis_10_lists = [[i == j for j in range(10)] for i in range(10)]
#obtain unique company tickers
comp_ticks = [i for i in df['company_ticker'].unique()]
#zip them together
ticks_TF_vis_lists = zip(comp_ticks, TF_vis_10_lists)

#create dropdown buttons
    #for the original figure, 'adj_close' was chosen as the default metric; indicate that here
comp_buttons = [{'label':comp, 'method':'update', 'args':[{
        'visible':TF_list}, {'title':f"{comp} - Adj Close"}]} for comp,TF_list in ticks_TF_vis_lists]

In [81]:
#Construct dropdown menus -- Price metric
#list the metrics, clean them up
metrics = {'open':'Open', 'high':'High', 'low':'Low', 'close':'Close', 'adj_close':'Adj Close',
          'volume':'Volume'}
#need to collect the data for each company for each metric
comp_metr_data = [df[df['company_ticker'] == comp][col] for comp in comp_ticks for col in metrics.keys()]
#zip the metrics & data together
zip_metr_comp_data = zip(metrics.values(), comp_metr_data)

#create dropdown buttons
    #when the figure is first initialized, the default company ticker is: "BRK-A", can add this
metr_buttons = [{'label':metr, 'method':'update', 'args':[{
        'y':data}, {'title':f"{comp_ticks[0]} - {metr}"}]} for metr, data in zip_metr_comp_data]

Now that the dropdowns & other features have been constructed, they can be added to the figure.

In [82]:
#Update the figure layout with the new features
fig.update_layout({'title':'Adjusted Close Over Time', 
                   'xaxis':{'rangeselector': {'buttons':time_range_buttons},
                           'rangeslider': {'visible':False},
                           'type':'date'},
                  'updatemenus':[{'type':'dropdown', 'direction':'down', 'x':1.2, 'y':1.0,
                                'showactive':True, 'buttons':comp_buttons,
                                  #use the index of the default company ticker (BRK-A)
                                 'active':0},
                                {'type':'dropdown', 'direction':'down', 'x':1.2, 'y':0.6,
                                'showactive':True, 'buttons':metr_buttons,
                                 #use the default metric (adj_close)
                                'active':list(metrics.keys()).index('adj_close')}],
                  'hovermode':'x unified'})

fig.show();